In [1]:
import pickle
import numpy as np
import pandas as pd
import os
from iblatlas.atlas import BrainRegions
from datetime import datetime
from one.api import ONE

one = ONE()

In [2]:
prefix = '/home/ines/repositories/'

# Load data

In [3]:
import brainwidemap
bwm_df = brainwidemap.bwm_query()
n_sessions = bwm_df['eid'].unique().shape[0]
n_insertions = bwm_df['pid'].unique().shape[0]
print(f'{n_sessions} sessions with {n_insertions} individual neuropixel recordings')
bwm_pid = bwm_df['pid'].unique()

Loading bwm_query results from fixtures/2023_12_bwm_release.csv
459 sessions with 699 individual neuropixel recordings


## LDA axis

In [4]:
data_path = prefix + 'representation_learning_variability/paper-individuality/clustering/'
lda = pd.read_pickle(data_path + 'mouse_LDA_5_bins_cut18-06-2026')
lda = lda.rename(columns={0: 'lda_1'})

lda_eid = lda.loc[lda['session'].isin(list(bwm_df.eid)), 'session']
lda_pid = bwm_df.loc[bwm_df['eid'].isin(lda_eid), 'pid']

print(f'Matching sessions: {len(lda_eid)}')
print(f'Matching insertions: {len(lda_pid)}')

Matching sessions: 244
Matching insertions: 380


In [5]:
regions = BrainRegions()

def get_region_acronym(col_name):
    raw_acronym = col_name.split('_neuron_')[0]
    allen_ids = regions.acronym2id(raw_acronym)
    beryl_ids = regions.remap(allen_ids, source_map='Allen', target_map='Beryl')
    return regions.id2acronym(beryl_ids)[0]

def extract_trial_spike_counts(df, col_name, events, pre=0.5, post=1.0):
    spike_counts = []
    valid_indices = []
    expected_bins = int((pre + post) * 60)
    
    for idx, event in enumerate(events):
        start = event - pre
        end = event + post
        window_data = df.loc[(df['Bin'] >= start) & (df['Bin'] < end), col_name]
        
        if len(window_data) == expected_bins:
            spike_counts.append(window_data.sum())
            valid_indices.append(idx)
    
    return np.array(spike_counts), np.array(valid_indices)

In [ ]:
path_dir = prefix + 'representation_learning_variability/paper-individuality/data/neuron_files/'
MIN_TRIALS = 10
MAX_TRIALS = 200
WINDOW_DURATION = 1.5  # seconds (0.5s pre + 1.0s post)

summary_records = []

for pid in lda_pid:
    filepath = os.path.join(path_dir, f'states_neurons_file_{pid}')
    
    try:
        with open(filepath, 'rb') as f:
            raw_data = pickle.load(f)
        
        state_data = raw_data.dropna(subset=['Bin', 'most_likely_states']).reset_index(drop=True)
        if state_data.empty:
            continue
        
        pid_name = state_data['session_pid'].iloc[0]
        session_name = state_data['session'].iloc[0]
        
        def get_condition(row):
            stim_side = 'Right' if row['correct'] == 1 and row['choice'] == 'left' else 'Left'
            return f"{stim_side}_{row['contrast']}"
        
        state_data['condition'] = state_data.apply(get_condition, axis=1)
        
        all_events = np.array(state_data['goCueTrigger_times'].unique())
        events_subset = all_events[:MAX_TRIALS]
        
        trial_metadata = state_data.drop_duplicates(subset=['goCueTrigger_times'])
        trial_conditions = trial_metadata.loc[
            trial_metadata['goCueTrigger_times'].isin(events_subset), 'condition'
        ].values
        
        spike_columns = [col for col in state_data.columns if '_spike_count' in col]
        
        for col in spike_columns:
            try:
                area = get_region_acronym(col)
            except:
                continue
            
            spike_counts, valid_idx = extract_trial_spike_counts(
                state_data, col, events_subset, pre=0.5, post=1.0
            )
            
            if len(spike_counts) < MIN_TRIALS:
                continue
            
            firing_rates = spike_counts / WINDOW_DURATION
            valid_conditions = trial_conditions[valid_idx]
            
            for condition in np.unique(valid_conditions):
                cond_mask = valid_conditions == condition
                cond_rates = firing_rates[cond_mask]
                
                if len(cond_rates) < 2:
                    continue
                
                record = {
                    'pid': pid_name,
                    'session': session_name,
                    'neuron_id': col,
                    'area': area,
                    'condition': condition,
                    'n_trials': len(cond_rates),
                    'mean_firing_rate_hz': np.mean(cond_rates),
                    'std_firing_rate_hz': np.std(cond_rates),
                }
                summary_records.append(record)
        
        print(f'Processed {pid_name}')
        
    except Exception as e:
        print(f'Error processing {filepath}: {e}')

summary_df = pd.DataFrame(summary_records)
print(f'\\nFinal dataset: {len(summary_df)} neuron-condition pairs')

In [ ]:
save_path = prefix + 'representation_learning_variability/paper-individuality/4_mice/'
current_date = datetime.now().strftime('%d-%m-%Y')
filename = f'firing_rates_per_neuron_condition_{current_date}'

summary_df.to_parquet(save_path + filename, compression='gzip')
print(f'Saved to {filename}')
print(f'\\nDataframe shape: {summary_df.shape}')
print(f'Columns: {summary_df.columns.tolist()}')
print(f'\\nSample:')
print(summary_df.head())